In [90]:
import pandas as pd
import numpy as np
import re
import string
import pickle

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, hamming_loss, f1_score


In [91]:
messages = pd.read_csv('../dataset/disaster_messages.csv')

categories = pd.read_csv('../dataset/disaster_categories.csv')

In [92]:
messages.head()

,id,message,original,genre
0,2,Weather update - a cold front from Cuba that c...,Un front froid se retrouve sur Cuba ce matin. ...,direct
1,7,Is the Hurricane over or is it not over,Cyclone nan fini osinon li pa fini,direct
2,8,Looking for someone but no name,"Patnm, di Maryani relem pou li banm nouvel li ...",direct
3,9,UN reports Leogane 80-90 destroyed. Only Hospi...,UN reports Leogane 80-90 destroyed. Only Hospi...,direct
4,12,"says: west side of Haiti, rest of the country ...",facade ouest d Haiti et le reste du pays aujou...,direct


In [93]:
categories.head()

,id,categories
0,2,related-1;request-0;offer-0;aid_related-0;medi...
1,7,related-1;request-0;offer-0;aid_related-1;medi...
2,8,related-1;request-0;offer-0;aid_related-0;medi...
3,9,related-1;request-1;offer-0;aid_related-1;medi...
4,12,related-1;request-0;offer-0;aid_related-0;medi...


In [94]:
df = messages.merge(
    categories,
    on="id"
)

In [95]:
df.head()

,id,message,original,genre,categories
0,2,Weather update - a cold front from Cuba that c...,Un front froid se retrouve sur Cuba ce matin. ...,direct,related-1;request-0;offer-0;aid_related-0;medi...
1,7,Is the Hurricane over or is it not over,Cyclone nan fini osinon li pa fini,direct,related-1;request-0;offer-0;aid_related-1;medi...
2,8,Looking for someone but no name,"Patnm, di Maryani relem pou li banm nouvel li ...",direct,related-1;request-0;offer-0;aid_related-0;medi...
3,9,UN reports Leogane 80-90 destroyed. Only Hospi...,UN reports Leogane 80-90 destroyed. Only Hospi...,direct,related-1;request-1;offer-0;aid_related-1;medi...
4,12,"says: west side of Haiti, rest of the country ...",facade ouest d Haiti et le reste du pays aujou...,direct,related-1;request-0;offer-0;aid_related-0;medi...


In [96]:
categories_column = df["categories"]

In [97]:
category_names = categories_column.iloc[0].split(";")

In [98]:
category_names

['related-1',
 'request-0',
 'offer-0',
 'aid_related-0',
 'medical_help-0',
 'medical_products-0',
 'search_and_rescue-0',
 'security-0',
 'military-0',
 'child_alone-0',
 'water-0',
 'food-0',
 'shelter-0',
 'clothing-0',
 'money-0',
 'missing_people-0',
 'refugees-0',
 'death-0',
 'other_aid-0',
 'infrastructure_related-0',
 'transport-0',
 'buildings-0',
 'electricity-0',
 'tools-0',
 'hospitals-0',
 'shops-0',
 'aid_centers-0',
 'other_infrastructure-0',
 'weather_related-0',
 'floods-0',
 'storm-0',
 'fire-0',
 'earthquake-0',
 'cold-0',
 'other_weather-0',
 'direct_report-0']

In [99]:
category_colnames = [
    row.split("-")[0]
    for row in category_names
]

In [100]:
categories_column.head()

0    related-1;request-0;offer-0;aid_related-0;medi...
1    related-1;request-0;offer-0;aid_related-1;medi...
2    related-1;request-0;offer-0;aid_related-0;medi...
3    related-1;request-1;offer-0;aid_related-1;medi...
4    related-1;request-0;offer-0;aid_related-0;medi...
Name: categories, dtype: str

In [101]:
categories_expanded = categories_column.str.split(
    ";",
    expand=True
)

In [102]:
for column in categories_expanded:

    categories_expanded[column] = categories_expanded[column].apply(
        lambda x: int(x.split("-")[1])
    )

    categories_expanded[column] = categories_expanded[column].replace(2, 1)

In [103]:
categories_expanded.columns = category_colnames

In [104]:
df = df.drop(columns=["categories"])

In [105]:
df = pd.concat(
    [df, categories_expanded],
    axis=1
)

df = df.drop_duplicates()

df = df.dropna(
    subset=["message"]
)

df.head()

,id,message,original,genre,related,request,offer,aid_related,medical_help,medical_products,...,aid_centers,other_infrastructure,weather_related,floods,storm,fire,earthquake,cold,other_weather,direct_report
0,2,Weather update - a cold front from Cuba that c...,Un front froid se retrouve sur Cuba ce matin. ...,direct,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,7,Is the Hurricane over or is it not over,Cyclone nan fini osinon li pa fini,direct,1,0,0,1,0,0,...,0,0,1,0,1,0,0,0,0,0
2,8,Looking for someone but no name,"Patnm, di Maryani relem pou li banm nouvel li ...",direct,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,9,UN reports Leogane 80-90 destroyed. Only Hospi...,UN reports Leogane 80-90 destroyed. Only Hospi...,direct,1,1,0,1,0,1,...,0,0,0,0,0,0,0,0,0,0
4,12,"says: west side of Haiti, rest of the country ...",facade ouest d Haiti et le reste du pays aujou...,direct,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [106]:
df.head()

,id,message,original,genre,related,request,offer,aid_related,medical_help,medical_products,...,aid_centers,other_infrastructure,weather_related,floods,storm,fire,earthquake,cold,other_weather,direct_report
0,2,Weather update - a cold front from Cuba that c...,Un front froid se retrouve sur Cuba ce matin. ...,direct,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,7,Is the Hurricane over or is it not over,Cyclone nan fini osinon li pa fini,direct,1,0,0,1,0,0,...,0,0,1,0,1,0,0,0,0,0
2,8,Looking for someone but no name,"Patnm, di Maryani relem pou li banm nouvel li ...",direct,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,9,UN reports Leogane 80-90 destroyed. Only Hospi...,UN reports Leogane 80-90 destroyed. Only Hospi...,direct,1,1,0,1,0,1,...,0,0,0,0,0,0,0,0,0,0
4,12,"says: west side of Haiti, rest of the country ...",facade ouest d Haiti et le reste du pays aujou...,direct,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [107]:
def clean_text(text):

    text = text.lower()

    text = re.sub(r"http\S+", "", text)

    text = re.sub(r"@\w+", "", text)

    text = re.sub(r"#", "", text)

    text = text.translate(
        str.maketrans("", "", string.punctuation)
    )

    text = text.strip()

    return text

In [108]:
print(df.shape)
print(df["message"].isnull().sum())

(26215, 40)
0


In [109]:
df["clean_message"] = df["message"].apply(
    clean_text
)

In [110]:
valid_columns = []

for column in category_colnames:

    unique_values = df[column].nunique()

    positive_count = df[column].sum()

    if unique_values > 1 and positive_count > 100:

        valid_columns.append(column)

category_colnames = valid_columns

X = df["clean_message"]

y = df[category_colnames].astype(int)

In [111]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [112]:
tfidf = TfidfVectorizer(

    max_features=30000,

    ngram_range=(1,2),

    stop_words="english",

    sublinear_tf=True,

    min_df=2,

    max_df=0.90
)

In [113]:
X_train_tfidf = tfidf.fit_transform(
    X_train
)

X_test_tfidf = tfidf.transform(
    X_test
)

In [114]:
model = OneVsRestClassifier(

    LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        solver="liblinear",
    )

)

In [115]:
model.fit(
    X_train_tfidf,
    y_train.values
)

,"estimator estimator: estimator objectA regressor or a classifier that implements :term:`fit`.When a classifier is passed, :term:`decision_function` will be usedin priority and it will fallback to :term:`predict_proba` if it is notavailable.When a regressor is passed, :term:`predict` is used.",LogisticRegre...r='liblinear')
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation: the `n_classes`one-vs-rest problems are computed in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: 0.20 `n_jobs` default changed from 1 to None",None
,"verbose verbose: int, default=0The verbosity level, if non zero, progress messages are printed.Below 50, the output is sent to stderr. Otherwise, the output is sentto stdout. The frequency of the messages increases with the verbositylevel, reporting all iterations at 10. See :class:`joblib.Parallel` formore details... versionadded:: 1.1",0
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=

In [116]:
y_pred = model.predict(
    X_test_tfidf
)

In [117]:
print(
    classification_report(
        y_test.values,
        y_pred,
        target_names=category_colnames,
        zero_division=0
    )
)

                        precision    recall  f1-score   support

               related       0.91      0.81      0.86      3998
               request       0.61      0.72      0.66       891
                 offer       0.00      0.00      0.00        24
           aid_related       0.73      0.74      0.74      2164
          medical_help       0.40      0.64      0.49       435
      medical_products       0.32      0.57      0.41       279
     search_and_rescue       0.22      0.39      0.28       136
              security       0.10      0.15      0.12        96
              military       0.38      0.70      0.50       158
                 water       0.60      0.84      0.70       335
                  food       0.74      0.86      0.80       584
               shelter       0.57      0.78      0.65       468
              clothing       0.38      0.60      0.47        70
                 money       0.32      0.62      0.42       112
        missing_people       0.22      

In [118]:
hamming = hamming_loss(
    y_test,
    y_pred
)

micro_f1 = f1_score(
    y_test,
    y_pred,
    average="micro"
)

macro_f1 = f1_score(
    y_test,
    y_pred,
    average="macro"
)

weighted_f1 = f1_score(
    y_test,
    y_pred,
    average="weighted"
)

print("\nMODEL EVALUATION\n")

print(f"Hamming Loss: {hamming:.4f}")

print(f"Micro F1 Score: {micro_f1:.4f}")

print(f"Macro F1 Score: {macro_f1:.4f}")

print(f"Weighted F1 Score: {weighted_f1:.4f}")


MODEL EVALUATION

Hamming Loss: 0.0724
Micro F1 Score: 0.6406
Macro F1 Score: 0.4523
Weighted F1 Score: 0.6654


In [119]:
pickle.dump(
    model,
    open(
        "../server/artifacts/model.pkl",
        "wb"
    )
)

In [120]:
pickle.dump(
    tfidf,
    open(
        "../server/artifacts/vectorizer.pkl",
        "wb"
    )
)

In [121]:
pickle.dump(
    category_colnames,
    open(
        "../server/artifacts/label_columns.pkl",
        "wb"
    )
)

In [122]:
def get_priority(

    predicted_labels,
    message

):

    high_priority_labels = [

        "search_and_rescue",
        "medical_help",
        "death",
        "water",
        "food",
        "shelter"

    ]

    medium_priority_labels = [

        "floods",
        "storm",
        "earthquake",
        "fire",
        "buildings",
        "electricity",
        "transport"

    ]

    urgency_keywords = [

        "urgent",
        "immediately",
        "trapped",
        "dying",
        "injured",
        "collapsed",
        "starving",
        "critical",
        "help",
        "rescue"

    ]

    high_score = 0

    medium_score = 0

    for label in predicted_labels:

        if label in high_priority_labels:

            high_score += 2

        elif label in medium_priority_labels:

            medium_score += 1

    message = message.lower()

    urgency_score = 0

    for word in urgency_keywords:

        if word in message:

            urgency_score += 1

    total_score = (
        high_score +
        medium_score +
        urgency_score
    )

    if total_score >= 6:

        return "HIGH"

    elif total_score >= 3:

        return "MEDIUM"

    else:

        return "LOW"

In [123]:
sample_message = "Free fire game destroyed my career."

In [124]:
clean_sample = clean_text(
    sample_message
)

In [125]:
sample_vector = tfidf.transform(
    [clean_sample]
)

In [126]:
probabilities = model.predict_proba(
    sample_vector
)

threshold = 0.75

ignore_labels = [

    "related",
    "aid_related",
    "request"

]

display_ignore_labels = [

    "direct_report",
    "other_aid",
    "weather_related",
    "infrastructure_related",
    "other_weather",
    "other_infrastructure"

]

predicted_labels = []

for label, prob in zip(category_colnames, probabilities[0]):

    if prob >= threshold and label not in ignore_labels and label not in display_ignore_labels:
        predicted_labels.append(label)

In [127]:
priority = get_priority(

    predicted_labels,
    message,

)

In [128]:
print("\nPredicted Emergency Categories:\n")

for label in predicted_labels:

    print(f"- {label}")
    
print(f"\nAssigned Priority: {priority}")


Predicted Emergency Categories:

- buildings

Assigned Priority: LOW


In [129]:
test_cases = [

    "People are trapped inside the collapsed building and urgently need medical assistance and clean water",

    "There has been no electricity for three days and families are struggling without food supplies",

    "I am watching a movie at home while eating pizza during heavy rain outside",

    "Children in the camp are starving and require immediate food and medical support",

    "Roads are flooded and rescue teams cannot reach the village because bridges are damaged",

    "My internet is slow and I cannot play online games tonight",

    "Huge fire accident near the hospital and many people are injured",

    "Families lost their homes after the cyclone and need shelter immediately",

    "Heavy earthquake destroyed several buildings and rescue teams are required urgently",

    "Water supply has been completely cut off and people are suffering badly"
]

In [130]:
for index, message in enumerate(test_cases, start=1):

    clean_sample = clean_text(message)

    sample_vector = tfidf.transform(
        [clean_sample]
    )

    probabilities = model.predict_proba(
        sample_vector
    )

    threshold = 0.75

    ignore_labels = [

        "related",
        "aid_related",
        "request"

    ]

    display_ignore_labels = [

    "direct_report",
    "other_aid",
    "weather_related",
    "infrastructure_related",
    "other_weather",
    "other_infrastructure"

    ]
    

    predicted_labels = []

    for label, prob in zip(
        category_colnames,
        probabilities[0]
    ):

        
        if prob >= threshold and label not in ignore_labels and label not in display_ignore_labels:

            predicted_labels.append(label)
    
    if not predicted_labels:

        print("No disaster-related categories detected.")

    priority = get_priority(

        predicted_labels,
        message

    )

    print("\n====================================")

    print(f"\nTEST CASE {index}")

    print(f"\nMESSAGE:\n{message}")

    print("\nPREDICTED LABELS:\n")

    for label, prob in zip(
    category_colnames,
    probabilities[0]
):

        if label in predicted_labels:

            print(f"- {label}: {prob:.2f}")

    print(f"\nPRIORITY LEVEL: {priority}")

    print("\n====================================")



TEST CASE 1

MESSAGE:
People are trapped inside the collapsed building and urgently need medical assistance and clean water

PREDICTED LABELS:

- medical_help: 0.95
- medical_products: 0.81
- search_and_rescue: 0.82
- water: 0.94
- buildings: 0.93

PRIORITY LEVEL: HIGH



TEST CASE 2

MESSAGE:
There has been no electricity for three days and families are struggling without food supplies

PREDICTED LABELS:

- medical_products: 0.90
- food: 0.99
- electricity: 0.94

PRIORITY LEVEL: MEDIUM



TEST CASE 3

MESSAGE:
I am watching a movie at home while eating pizza during heavy rain outside

PREDICTED LABELS:

- storm: 0.96

PRIORITY LEVEL: LOW



TEST CASE 4

MESSAGE:
Children in the camp are starving and require immediate food and medical support

PREDICTED LABELS:

- medical_help: 0.96
- medical_products: 0.82
- food: 1.00

PRIORITY LEVEL: MEDIUM



TEST CASE 5

MESSAGE:
Roads are flooded and rescue teams cannot reach the village because bridges are damaged

PREDICTED LABELS:

- search_